# Diagnose SOEN Training — Why is the model predicting all-clear?

Systematic checks for each potential failure point.

In [ ]:
import numpy as np
import h5py
import yaml
from pathlib import Path

SOEN_H5 = Path("soen_training/datasets/pca8_disruption_seq2seq_2class.h5")
CONFIG_PATH = Path("soen_training/training_configs/training_config_pca8_disruption_seq2seq_2class.yaml")

## Check 1: H5 file contents — shapes, dtypes, class balance

In [ ]:
print(f"=== H5: {SOEN_H5} ===\n")
with h5py.File(SOEN_H5, "r") as f:
    for split in ("train", "val", "test"):
        g = f[split]
        print(f"--- {split} ---")
        for key in g.keys():
            ds = g[key]
            print(f"  {key}: shape={ds.shape}, dtype={ds.dtype}")

        data = np.asarray(g["data"])
        labels = np.asarray(g["labels"])
        mask = np.asarray(g["target_mask"]) if "target_mask" in g else None
        N, T, D = data.shape

        # Label distribution
        unique, counts = np.unique(labels, return_counts=True)
        total_ts = labels.size
        print(f"  Labels: unique={unique}, counts={counts}, total={total_ts}")
        for u, c in zip(unique, counts):
            print(f"    class {u}: {c} ({c/total_ts*100:.1f}%)")

        # Mask analysis
        if mask is not None:
            n_valid = int(mask.sum())
            n_masked = int((~mask).sum())
            labels_in_valid = labels[mask]
            pos_in_valid = int((labels_in_valid == 1).sum())
            neg_in_valid = int((labels_in_valid == 0).sum())
            print(f"  target_mask: valid={n_valid}, masked={n_masked}")
            print(f"    WITHIN valid: pos={pos_in_valid} ({pos_in_valid/max(n_valid,1)*100:.1f}%), "
                  f"neg={neg_in_valid} ({neg_in_valid/max(n_valid,1)*100:.1f}%)")
        else:
            print(f"  target_mask: NOT PRESENT IN H5")

        # Data stats
        print(f"  data: min={data.min():.4f}, max={data.max():.4f}, "
              f"mean={data.mean():.4f}, std={data.std():.4f}")

        # Per-sequence stats
        seq_has_pos = (labels.sum(axis=1) > 0).sum()
        print(f"  Sequences with any positive label: {seq_has_pos}/{N}")
        print()

## Check 2: Does the YAML config have `label_mask_key`? Does installed soenre2 support it?

In [ ]:
cfg = yaml.safe_load(CONFIG_PATH.read_text())

print("=== Config: label_mask_key ===")
print(f"  data.label_mask_key: {cfg.get('data', {}).get('label_mask_key', 'NOT SET')}")
print(f"  data.num_classes:    {cfg.get('data', {}).get('num_classes')}")
print(f"  training.mapping:    {cfg.get('training', {}).get('mapping')}")
print(f"  training.losses:     {cfg.get('training', {}).get('losses')}")
print()

# Check if installed soen_toolkit DataConfig has label_mask_key
import subprocess, sys

_SOEN_SRC_CANDIDATES = [
    Path("/home/idies/workspace/Temporary/dpark1/scratch/soenhardware/soen-toolkit/src"),
    Path("/home/idies/workspace/Temporary/dpark1/scratch/SOEN/soenre2/src"),
    Path("/Users/davidpark/Documents/Cursor/soenhardware/soen-toolkit/src"),
]
SOEN_SRC = None
for _c in _SOEN_SRC_CANDIDATES:
    if (_c / "soen_toolkit" / "__init__.py").exists():
        SOEN_SRC = _c
        break

# Find working python
import shutil
_PYTHON_CANDIDATES = [
    SOEN_SRC.parent / ".venv" / "bin" / "python" if SOEN_SRC else "",
    Path("/home/idies/workspace/Temporary/dpark1/scratch/conda/conda_envs/soen/bin/python"),
    sys.executable,
]
SOEN_PYTHON = None
for _p in _PYTHON_CANDIDATES:
    _p = Path(str(_p))
    if _p.exists():
        _test = subprocess.run(
            [str(_p), "-c", "import soen_toolkit; print('OK')"],
            env={"PYTHONPATH": str(SOEN_SRC), "PATH": "/usr/bin:/bin"},
            capture_output=True, text=True)
        if _test.returncode == 0:
            SOEN_PYTHON = _p
            break

print("=== Installed soen_toolkit DataConfig fields ===")
r = subprocess.run(
    [str(SOEN_PYTHON), "-c",
     "import dataclasses; "
     "from soen_toolkit.training.configs.config_classes import DataConfig; "
     "fields = [f.name for f in dataclasses.fields(DataConfig)]; "
     "print('label_mask_key' in fields); "
     "print('HAS label_mask_key' if 'label_mask_key' in fields else 'MISSING label_mask_key'); "
     "print('All fields:', sorted(fields))"],
    env={"PYTHONPATH": str(SOEN_SRC), "PATH": "/usr/bin:/bin"},
    capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("stderr:", r.stderr[:300])

## Check 3: Does cross_entropy actually receive and use target_mask?

Trace what the loss function gets by checking the source code of the installed version.

In [ ]:
# Check if the installed cross_entropy accepts target_mask
# AND if the dataloader actually passes it
r = subprocess.run(
    [str(SOEN_PYTHON), "-c", """
import inspect
from soen_toolkit.training.losses import cross_entropy

# Check cross_entropy signature
sig = inspect.signature(cross_entropy)
print("cross_entropy signature:", sig)
print("Parameters:", list(sig.parameters.keys()))
print()

# Check if there's a JAX version too
try:
    from soen_toolkit.utils.port_to_jax.jax_training.losses import jax_loss_registry
    print("JAX loss registry keys:", list(jax_loss_registry.keys()) if hasattr(jax_loss_registry, 'keys') else 'not a dict')
except Exception as e:
    print(f"JAX loss check failed: {e}")

# Check what the dataloader returns
try:
    from soen_toolkit.training.configs.config_classes import DataConfig
    import dataclasses
    fields = {f.name: f.default for f in dataclasses.fields(DataConfig)}
    print()
    print("DataConfig.label_mask_key default:", fields.get('label_mask_key', 'FIELD NOT FOUND'))
    print("DataConfig.require_label_mask default:", fields.get('require_label_mask', 'FIELD NOT FOUND'))
except Exception as e:
    print(f"DataConfig check failed: {e}")

# Check if cross_entropy in JAX backend handles masks
try:
    import importlib
    jax_losses = importlib.import_module("soen_toolkit.utils.port_to_jax.jax_training.losses")
    src = inspect.getsource(jax_losses)
    # Find cross_entropy-related functions
    for line_no, line in enumerate(src.split('\\n')):
        if 'cross_entropy' in line.lower() and ('def ' in line or 'mask' in line.lower() or 'target_mask' in line.lower()):
            print(f"  JAX losses L{line_no}: {line.strip()}")
except Exception as e:
    print(f"JAX losses source check failed: {e}")
"""],
    env={"PYTHONPATH": str(SOEN_SRC), "PATH": "/usr/bin:/bin"},
    capture_output=True, text=True)

print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr[:500])

## Check 4: Model output shape and T_out vs T_label alignment

In [ ]:
# Check model output shape by running a single batch
r = subprocess.run(
    [str(SOEN_PYTHON), "-c", f"""
import numpy as np, h5py, torch, json
from soen_toolkit.core.soen_model_core import SOENModelCore

# Load a small batch from train
with h5py.File({str(SOEN_H5.resolve())!r}, "r") as f:
    X = torch.from_numpy(np.asarray(f["train"]["data"][:4]))  # (4, 279, 224)
    labels = np.asarray(f["train"]["labels"][:4])              # (4, 279)

print(f"Input shape: {{X.shape}}")
print(f"Labels shape: {{labels.shape}}")

# Load the model
import glob
model_path = {str(Path("soen_training/model_specs/224IN_28H_24SQUID_2OffchipLinear_blockdiag.soen").resolve())!r}
model = SOENModelCore.load(model_path)
model.eval()

with torch.no_grad():
    out = model(X)
    if isinstance(out, tuple):
        print(f"Output is tuple of length {{len(out)}}")
        for i, o in enumerate(out):
            if hasattr(o, 'shape'):
                print(f"  out[{{i}}]: shape={{o.shape}}, dtype={{o.dtype}}")
            else:
                print(f"  out[{{i}}]: type={{type(o).__name__}}")
        out_tensor = out[0]
    else:
        out_tensor = out
        print(f"Output shape: {{out_tensor.shape}}")

print(f"\\nOutput shape: {{out_tensor.shape}}")
print(f"Labels shape: {{labels.shape}}")
print(f"T mismatch: output T={{out_tensor.shape[1]}}, labels T={{labels.shape[1]}}")

# Check logit values
logits = out_tensor.numpy()
if logits.ndim == 3:
    print(f"\\nLogit stats (raw model, no training):")
    print(f"  ch0 (clear):     mean={{logits[:,:,0].mean():.4f}}, std={{logits[:,:,0].std():.4f}}")
    print(f"  ch1 (disruptive): mean={{logits[:,:,1].mean():.4f}}, std={{logits[:,:,1].std():.4f}}")
    print(f"  argmax distribution: {{dict(zip(*np.unique(logits.argmax(axis=-1), return_counts=True)))}}")
    print(f"  Sample logits [0, -5:]: {{logits[0, -5:, :].tolist()}}")
"""],
    env={"PYTHONPATH": str(SOEN_SRC), "PATH": "/usr/bin:/bin"},
    capture_output=True, text=True)

print(r.stdout)
if r.stderr:
    # Filter out the common solver warning
    lines = [l for l in r.stderr.split("\n") if "g_table" not in l and l.strip()]
    if lines:
        print("STDERR:", "\n".join(lines[:10]))

## Check 5: How does the JAX training backend compute the loss? Does it use the mask?

Check the actual loss computation path in the JAX backend trainer.

In [ ]:
# Check the JAX backend's loss computation — does it pass mask to the loss?
import os as _os
_env = {**_os.environ, "PYTHONPATH": str(SOEN_SRC)}

r = subprocess.run(
    [str(SOEN_PYTHON), "-c", """
import inspect

# 1. Check JAX trainer's _compute_batch_loss_core
try:
    from soen_toolkit.utils.port_to_jax.jax_training.trainer import Trainer
    src = inspect.getsource(Trainer._compute_batch_loss_core)
    print("=== Trainer._compute_batch_loss_core (key lines) ===")
    for i, line in enumerate(src.split('\\n')):
        if any(kw in line.lower() for kw in ['mask', 'target_mask', 'valid_mask', 'label_mask', 'loss_fn', 'cross_entropy']):
            print(f"  L{i}: {line.rstrip()}")
except Exception as e:
    print(f"Failed: {e}")

print()

# 2. Check JAX cross_entropy implementation
try:
    from soen_toolkit.utils.port_to_jax.jax_training import losses as jax_losses
    for name in dir(jax_losses):
        if 'cross_entropy' in name.lower():
            fn = getattr(jax_losses, name)
            if callable(fn):
                print(f"=== jax_losses.{name} ===")
                src = inspect.getsource(fn)
                for i, line in enumerate(src.split('\\n')[:30]):
                    print(f"  {line}")
                print()
except Exception as e:
    print(f"JAX losses failed: {e}")

# 3. Check how the dataloader constructs batches — mask related
try:
    from soen_toolkit.training.data import datasets
    src = inspect.getsource(datasets)
    print("=== Dataset source (mask-related lines) ===")
    for i, line in enumerate(src.split('\\n')):
        if any(kw in line.lower() for kw in ['label_mask', 'target_mask', 'mask_key', '__getitem__']):
            print(f"  L{i}: {line.rstrip()}")
except Exception as e:
    print(f"Dataset source failed: {e}")
"""],
    env=_env, capture_output=True, text=True)

print(r.stdout[:3000])
if r.stderr:
    lines = [l for l in r.stderr.split("\n") if l.strip() and "g_table" not in l]
    if lines:
        print("STDERR:", "\n".join(lines[:5]))

## Check 6: Visualize a few disruptive sequences — labels, mask, data

In [ ]:
import matplotlib.pyplot as plt

with h5py.File(SOEN_H5, "r") as f:
    labels = np.asarray(f["train"]["labels"])
    mask = np.asarray(f["train"]["target_mask"])
    data = np.asarray(f["train"]["data"])

# Find disruptive sequences (have positive labels)
disruptive_idx = np.where(labels.any(axis=1))[0]
clear_idx = np.where(~labels.any(axis=1))[0]
print(f"Train: {len(disruptive_idx)} disruptive, {len(clear_idx)} clear sequences")

fig, axes = plt.subplots(4, 2, figsize=(16, 12))

for col, (name, indices) in enumerate([("Disruptive", disruptive_idx), ("Clear", clear_idx)]):
    if len(indices) == 0:
        continue
    idx = indices[0]

    # Row 0: Labels
    axes[0, col].plot(labels[idx], 'r-', linewidth=1)
    axes[0, col].set_title(f"{name} seq #{idx} — Labels")
    axes[0, col].set_ylim(-0.1, 1.1)

    # Row 1: Target mask
    axes[1, col].plot(mask[idx].astype(float), 'b-', linewidth=1)
    axes[1, col].set_title("Target mask (True=valid for loss)")
    axes[1, col].set_ylim(-0.1, 1.1)

    # Row 2: Labels * mask (what the loss actually sees)
    effective = labels[idx] * mask[idx]
    axes[2, col].plot(effective, 'g-', linewidth=1)
    axes[2, col].set_title("Effective labels (labels * mask)")
    axes[2, col].set_ylim(-0.1, 1.1)

    n_pos = int(labels[idx][mask[idx]].sum()) if mask[idx].any() else 0
    n_neg = int((labels[idx][mask[idx]] == 0).sum()) if mask[idx].any() else 0
    axes[2, col].text(0.02, 0.8, f"pos={n_pos}, neg={n_neg}",
                      transform=axes[2, col].transAxes, fontsize=10)

    # Row 3: First 4 input channels
    for ch in range(min(4, data.shape[2])):
        axes[3, col].plot(data[idx, :, ch], alpha=0.5, label=f"ch{ch}")
    axes[3, col].set_title("Input data (first 4 of 224 channels)")
    axes[3, col].legend(fontsize=7)
    axes[3, col].set_xlabel("SOEN timestep (of 279)")

plt.tight_layout()
plt.show()

# Summary stats
print(f"\nTrain set summary:")
print(f"  Total sequences: {len(labels)}")
print(f"  Disruptive: {len(disruptive_idx)} ({len(disruptive_idx)/len(labels)*100:.1f}%)")
print(f"  Total valid timesteps: {int(mask.sum())}")
print(f"  Positive in valid: {int((labels[mask]==1).sum())} ({int((labels[mask]==1).sum())/max(int(mask.sum()),1)*100:.1f}%)")
print(f"  Negative in valid: {int((labels[mask]==0).sum())} ({int((labels[mask]==0).sum())/max(int(mask.sum()),1)*100:.1f}%)")

## Check 7: What does the generated YAML config actually look like?

In [ ]:
print("=== Full YAML config ===\n")
print(CONFIG_PATH.read_text())

## Summary: What to look for

After running all checks:

1. **Check 1**: Is the H5 balanced? Are `target_mask`'d positive/negative counts ~50/50?
2. **Check 2**: Does `label_mask_key` exist in the installed DataConfig? If MISSING, the mask is **silently ignored**.
3. **Check 3**: Does the cross_entropy loss accept `target_mask`? Does the JAX version?
4. **Check 4**: Is T_out (model) = T_label (data)? If 280 vs 279, labels are **misaligned** after flatten.
5. **Check 5**: Does the JAX trainer pass the mask to the loss? Or does it only pass `(outputs, targets)`?
6. **Check 6**: Visual sanity — do labels/mask/data look correct?
7. **Check 7**: Does the YAML have all expected fields?

**Most likely failure**: `label_mask_key` is not supported → mask is ignored → unbalanced loss → trivial prediction.